In [ ]:
import io
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

# --- Widgets ---
upload = widgets.FileUpload(accept='image/*', multiple=False, description='Upload Image')
cmap_selector = widgets.Dropdown(options=plt.colormaps(), value='gray', description='Colormap:')
scale_selector = widgets.Dropdown(options=['linear', 'log'], value='log', description='FFT Scale:')
zoom_slider = widgets.IntSlider(value=100, min=10, max=500, step=10, description='FFT Crop:', continuous_update=True)
filter_type_selector = widgets.Dropdown(options=['None', 'Low-pass', 'High-pass'], value='None', description='Filter:')
filter_radius_slider = widgets.IntSlider(value=50, min=5, max=250, step=5, description='Radius:', continuous_update=True)

controls = widgets.VBox([upload, cmap_selector, scale_selector, zoom_slider, filter_type_selector, filter_radius_slider])
display(controls)

out_original = widgets.Output()
out_fft = widgets.Output()
out_filtered = widgets.Output()
display(widgets.HBox([out_original, out_fft, out_filtered]))

# --- Circular mask function ---
def circular_mask(shape, radius, filter_type):
    rows, cols = shape
    crow, ccol = rows//2, cols//2
    Y, X = np.ogrid[:rows, :cols]
    distance = np.sqrt((X - ccol)**2 + (Y - crow)**2)
    if filter_type == 'Low-pass':
        mask = distance <= radius
    elif filter_type == 'High-pass':
        mask = distance >= radius
    else:
        mask = np.ones(shape, dtype=bool)
    return mask

# --- Update function with error handling ---
def update_plot(change=None):
    try:
        if upload.value:
            uploaded_file = list(upload.value.values())[0]
            content = uploaded_file['content']
            img = Image.open(io.BytesIO(content)).convert('L')
            img_array = np.array(img)

            fft = np.fft.fft2(img_array)
            fft_shifted = np.fft.fftshift(fft)
            magnitude = np.abs(fft_shifted)
            if scale_selector.value == 'log':
                magnitude_display = np.log1p(magnitude)
            else:
                magnitude_display = magnitude.copy()

            mask = circular_mask(fft_shifted.shape, filter_radius_slider.value, filter_type_selector.value)
            fft_filtered = fft_shifted * mask

            center_x, center_y = fft_filtered.shape[1]//2, fft_filtered.shape[0]//2
            crop_size = zoom_slider.value
            x_min = max(center_x - crop_size//2, 0)
            x_max = min(center_x + crop_size//2, fft_filtered.shape[1])
            y_min = max(center_y - crop_size//2, 0)
            y_max = min(center_y + crop_size//2, fft_filtered.shape[0])
            magnitude_cropped = np.abs(fft_filtered[y_min:y_max, x_min:x_max])
            if scale_selector.value == 'log':
                magnitude_cropped = np.log1p(magnitude_cropped)

            img_filtered = np.fft.ifft2(np.fft.ifftshift(fft_filtered)).real

            with out_original:
                clear_output(wait=True)
                plt.figure(figsize=(4,4))
                plt.imshow(img_array, cmap=cmap_selector.value)
                plt.title('Original Image')
                plt.axis('off')
                plt.show()

            with out_fft:
                clear_output(wait=True)
                plt.figure(figsize=(4,4))
                plt.imshow(magnitude_cropped, cmap=cmap_selector.value)
                plt.title('FFT Magnitude')
                plt.axis('off')
                plt.show()

            with out_filtered:
                clear_output(wait=True)
                plt.figure(figsize=(4,4))
                plt.imshow(img_filtered, cmap=cmap_selector.value)
                plt.title('Filtered Image')
                plt.axis('off')
                plt.show()
    except Exception as e:
        with out_original:
            clear_output(wait=True)
            print(f"Error: {e}")

# --- Observe changes ---
upload.observe(update_plot, names='value')
cmap_selector.observe(update_plot, names='value')
scale_selector.observe(update_plot, names='value')
zoom_slider.observe(update_plot, names='value')
filter_type_selector.observe(update_plot, names='value')
filter_radius_slider.observe(update_plot, names='value')
